# Pidex — Pi Package Ecosystem Exploration

Data source: npm registry keyword `pi-package`


In [ ]:
import json, re
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import numpy as np

sns.set_theme(style='whitegrid')
DATA = Path('../data')

In [ ]:
# Load enriched packages
records = []
for f in (DATA / 'enriched').glob('*.json'):
    records.append(json.loads(f.read_text()))

df = pd.DataFrame(records)
print(f"Loaded {len(df)} packages")
df.head(3)

## 1. Overview

In [ ]:
# Infer type from keywords
def infer_type(keywords):
    for k in (keywords or []):
        if k in ('pi-extension', 'pi-skill', 'pi-theme', 'pi-prompt'):
            return k.replace('pi-', '')
    return 'unknown'

df['type'] = df['keywords'].apply(infer_type)
df['type'].value_counts().plot(kind='bar', title='Packages by type')
plt.tight_layout(); plt.show()
print(df['type'].value_counts())

In [ ]:
# Authors / publishers
publishers = df['publisher'].value_counts() if 'publisher' in df.columns else pd.Series()
publishers.head(20).plot(kind='barh', title='Top 20 publishers')
plt.tight_layout(); plt.show()

## 2. Download Distribution

In [ ]:
dl = df[['name', 'downloads_last_week', 'type']].dropna(subset=['downloads_last_week'])
dl = dl.sort_values('downloads_last_week', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dl['downloads_last_week'].plot(kind='hist', bins=50, ax=axes[0], title='Download distribution (linear)')
dl['downloads_last_week'].apply(lambda x: x+1).plot(
    kind='hist', bins=50, ax=axes[1], title='Download distribution (log scale)', logy=True)
plt.tight_layout(); plt.show()

print('Top 20 by downloads last week:')
print(dl.head(20)[['name', 'downloads_last_week', 'type']].to_string(index=False))

## 3. Download Trends Over Time

In [ ]:
# Build a wide DataFrame: index=week_end, columns=package_name
trend_rows = []
for _, row in df.iterrows():
    for entry in (row.get('download_trend') or []):
        trend_rows.append({'name': row['name'], 'week_end': entry['week_end'], 'downloads': entry['downloads']})

if trend_rows:
    tdf = pd.DataFrame(trend_rows)
    tdf['week_end'] = pd.to_datetime(tdf['week_end'])
    pivot = tdf.pivot_table(index='week_end', columns='name', values='downloads', aggfunc='sum').fillna(0)

    # Total ecosystem downloads per week (shows growth)
    total_weekly = pivot.sum(axis=1)
    fig, ax = plt.subplots(figsize=(14, 4))
    total_weekly.plot(ax=ax, title='Total pi ecosystem downloads per week')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.tight_layout(); plt.show()

    # Top N packages — trajectory comparison
    top_names = df.nlargest(15, 'downloads_last_week')['name'].tolist()
    top_names = [n for n in top_names if n in pivot.columns]
    fig, ax = plt.subplots(figsize=(14, 6))
    pivot[top_names].plot(ax=ax, title='Weekly downloads — top 15 packages')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.legend(loc='upper left', fontsize=7)
    plt.tight_layout(); plt.show()
else:
    print('No trend data — run enrich.py first')

In [ ]:
# Classify packages by trajectory: growing / declining / flat / new
if trend_rows:
    def classify_trend(series):
        s = series.dropna()
        if len(s) < 4: return 'new'
        first_half = s.iloc[:len(s)//2].mean()
        second_half = s.iloc[len(s)//2:].mean()
        if first_half == 0: return 'new'
        ratio = second_half / first_half
        if ratio > 1.5: return 'growing'
        if ratio < 0.5: return 'declining'
        return 'stable'

    trajectories = pivot.apply(classify_trend)
    print(trajectories.value_counts())
    trajectories.value_counts().plot(kind='bar', title='Package trajectory')
    plt.tight_layout(); plt.show()

## 4. Word Clouds

In [ ]:
def make_wordcloud(texts, title):
    combined = ' '.join(t for t in texts if isinstance(t, str))
    # Strip markdown formatting noise
    combined = re.sub(r'[#*`\[\](){}|>]', ' ', combined)
    wc = WordCloud(width=900, height=400, background_color='white', max_words=120).generate(combined)
    plt.figure(figsize=(14, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off'); plt.title(title); plt.tight_layout(); plt.show()

make_wordcloud(df['description'].tolist(), 'All packages — descriptions')
make_wordcloud(df['readme'].fillna('').tolist(), 'All packages — READMEs')

In [ ]:
# Per-type word clouds
for pkg_type in df['type'].unique():
    subset = df[df['type'] == pkg_type]
    if len(subset) >= 3:
        make_wordcloud(subset['description'].tolist(), f'{pkg_type} — descriptions ({len(subset)} pkgs)')

## 5. TF-IDF Similarity — Near-Duplicates

In [ ]:
# Combine description + first 500 chars of README for signal
df['text'] = (df['description'].fillna('') + ' ' + df['readme'].fillna('').str[:500])

vec = TfidfVectorizer(max_features=3000, stop_words='english', ngram_range=(1, 2))
tfidf = vec.fit_transform(df['text'])
sim = cosine_similarity(tfidf)
np.fill_diagonal(sim, 0)  # ignore self-similarity

# Find top near-duplicate pairs
pairs = []
n = len(df)
for i in range(n):
    for j in range(i+1, n):
        if sim[i, j] > 0.25:
            pairs.append({'pkg_a': df.iloc[i]['name'], 'pkg_b': df.iloc[j]['name'],
                          'similarity': round(sim[i, j], 3),
                          'dl_a': df.iloc[i]['downloads_last_week'],
                          'dl_b': df.iloc[j]['downloads_last_week']})

pairs_df = pd.DataFrame(pairs).sort_values('similarity', ascending=False)
print(f"Near-duplicate pairs (similarity > 0.25): {len(pairs_df)}")
print(pairs_df.head(30).to_string(index=False))

## 6. Cluster Analysis — Blockbusters vs Long Tail

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    import umap

    model = SentenceTransformer('all-MiniLM-L6-v2')
    print('Computing embeddings (CPU, may take a minute)...')
    embeddings = model.encode(df['text'].tolist(), show_progress_bar=True)

    reducer = umap.UMAP(n_neighbors=10, min_dist=0.1, metric='cosine', random_state=42)
    coords = reducer.fit_transform(embeddings)
    df['umap_x'] = coords[:, 0]
    df['umap_y'] = coords[:, 1]

    fig, ax = plt.subplots(figsize=(14, 10))
    type_colors = {'extension': 'steelblue', 'skill': 'darkorange', 'theme': 'green',
                   'prompt': 'crimson', 'unknown': 'grey'}
    for pkg_type, group in df.groupby('type'):
        ax.scatter(group['umap_x'], group['umap_y'],
                   c=type_colors.get(pkg_type, 'grey'), label=pkg_type, alpha=0.7, s=60)

    # Label top packages by download
    top20 = df.nlargest(20, 'downloads_last_week')
    for _, row in top20.iterrows():
        ax.annotate(row['name'].split('/')[-1], (row['umap_x'], row['umap_y']),
                    fontsize=6.5, ha='center', va='bottom')

    ax.legend(); ax.set_title('UMAP cluster plot — pi packages')
    plt.tight_layout(); plt.show()

except ImportError:
    print('sentence-transformers or umap-learn not installed — run: pip install sentence-transformers umap-learn')

In [ ]:
# Detect cluster patterns: blockbuster vs long tail
# Use TF-IDF clusters via KMeans as a quick proxy
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD

N_CLUSTERS = 12
svd = TruncatedSVD(n_components=50, random_state=42)
reduced = svd.fit_transform(tfidf)
km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(reduced)

# For each cluster: top terms, package count, total downloads, download distribution
feature_names = vec.get_feature_names_out()
order_centroids = km.cluster_centers_.argsort()[:, ::-1]

print('\n=== Cluster Summary ===\n')
cluster_summaries = []
for i in range(N_CLUSTERS):
    cluster_pkgs = df[df['cluster'] == i]
    top_terms = [feature_names[ind] for ind in order_centroids[i, :6]]
    dls = cluster_pkgs['downloads_last_week'].dropna()
    total_dl = dls.sum()
    max_dl = dls.max() if len(dls) else 0
    # Blockbuster ratio: top-1 / total — high means one dominant package
    blockbuster_ratio = (max_dl / total_dl) if total_dl > 0 else 0
    pattern = 'blockbuster' if blockbuster_ratio > 0.6 else ('contested' if blockbuster_ratio > 0.35 else 'long-tail')
    cluster_summaries.append({
        'cluster': i, 'n_pkgs': len(cluster_pkgs), 'top_terms': ', '.join(top_terms),
        'total_dl': int(total_dl), 'blockbuster_ratio': round(blockbuster_ratio, 2), 'pattern': pattern
    })
    print(f"Cluster {i:2d} | {len(cluster_pkgs):3d} pkgs | {pattern:12s} | {top_terms[:4]} | top pkg downloads: {int(max_dl)}")

clusters_df = pd.DataFrame(cluster_summaries)
print('\nPattern counts:', clusters_df['pattern'].value_counts().to_dict())

In [ ]:
# Drill into contested clusters — who are the competitors?
contested = clusters_df[clusters_df['pattern'].isin(['contested', 'blockbuster'])].sort_values('total_dl', ascending=False)

for _, c_row in contested.head(6).iterrows():
    cid = c_row['cluster']
    pkgs = df[df['cluster'] == cid][['name', 'description', 'downloads_last_week', 'type']]\
               .sort_values('downloads_last_week', ascending=False)
    print(f"\n--- Cluster {cid} [{c_row['pattern']}] — {c_row['top_terms']} ---")
    print(pkgs.head(8).to_string(index=False))

## 7. Findings Summary

In [ ]:
print('=== FINDINGS SUMMARY ===')
print(f"Total packages: {len(df)}")
print(f"By type: {df['type'].value_counts().to_dict()}")
print(f"Near-duplicate pairs (>0.25 TF-IDF): {len(pairs_df)}")
if 'trajectories' in dir():
    print(f"Trajectory breakdown: {trajectories.value_counts().to_dict()}")
print(f"Cluster patterns: {clusters_df['pattern'].value_counts().to_dict()}")
print(f"\nTop 5 near-dupes:")
print(pairs_df.head(5)[['pkg_a', 'pkg_b', 'similarity']].to_string(index=False))